In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [4]:
df = pd.read_csv("../data/raw/insurance.csv")

print("Shape:", df.shape)
print(df.head())

Shape: (1338, 7)
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [5]:
# Create a binary classification target
median_charge = df["charges"].median()

df["high_charge"] = (df["charges"] > median_charge).astype(int)

print("Median charge:", median_charge)
print("\nClass counts:")
print(df["high_charge"].value_counts())

Median charge: 9382.033

Class counts:
high_charge
1    669
0    669
Name: count, dtype: int64


In [6]:
# Select features and target
X = df[["age", "bmi", "children", "smoker"]].copy()
y = df["high_charge"]

# Convert smoker from text to numbers
X["smoker"] = X["smoker"].map({"no": 0, "yes": 1})

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   age     bmi  children  smoker
0   19  27.900         0       1
1   18  33.770         1       0
2   28  33.000         3       0
3   33  22.705         0       0
4   32  28.880         0       0

Target:
0    1
1    0
2    0
3    1
4    0
Name: high_charge, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1070, 4)
Testing data: (268, 4)


In [8]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression trained.")

Logistic Regression trained.


In [9]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", lr_accuracy)
print("Logistic Regression Precision:", lr_precision)
print("Logistic Regression Recall:", lr_recall)
print("Logistic Regression F1:", lr_f1)
print("\nConfusion matrix:")
print(lr_cm)

Logistic Regression Accuracy: 0.8992537313432836
Logistic Regression Precision: 0.8848920863309353
Logistic Regression Recall: 0.917910447761194
Logistic Regression F1: 0.9010989010989011

Confusion matrix:
[[118  16]
 [ 11 123]]


In [10]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)

print("Random Forest trained.")

Random Forest trained.


In [11]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1:", rf_f1)
print("\nConfusion matrix:")
print(rf_cm)

Random Forest Accuracy: 0.9291044776119403
Random Forest Precision: 0.967479674796748
Random Forest Recall: 0.8880597014925373
Random Forest F1: 0.9260700389105059

Confusion matrix:
[[130   4]
 [ 15 119]]


I optimized for recall instead of accuracy or precision because of what each mistake actually costs. If I miss someone who is truly high-cost, they never get flagged, nobody manages their risk early, and the insurer just absorbs a large, unexpected bill later. If I wrongly flag someone who is actually low-cost, the only cost is a bit of wasted outreach, like an unnecessary check-in call. Missing a real high-cost person is a much bigger problem than a false alarm, so I chose the model that catches more of the true high-cost people, even if that means a few more false alarms.

Logistic Regression caught 123 out of 134 high-cost people in my test set, missing only 11. Random Forest caught fewer, missing 15. Random Forest did score better on accuracy, precision, and F1, and it made far fewer false alarms overall (4 versus 16), so it is the more "correct" model in general. But since my main goal was catching as many high-cost people as possible, I chose Logistic Regression instead, because it directly performs better on the one thing I said mattered most.

If the business cared more about not wasting time on false alarms than about catching every high-cost case, Random Forest would be the better choice instead. My model choice follows from what I decided was the more expensive mistake, and someone could reasonably disagree with that assumption.